<h1>Tagging unmarked Ohio trails using data</h1>
<h4>12/4/24</h4>
<h4>Jacob Nicholson</h4>
<p>Many applications hikers use to tour American national parks rely on data from Open Street Maps, an open source editable map. These applications often translate Data from OSM(OpenSteetMaps) in a somewhat unfiltered way that displays private and unoffical trails as equal to recomened and offical trails maintained by the National Park. This reportedly causes users of trail apps derived from OSM such as All Trails to traverse these trails, putting themseleves and the wildilife in harms way. Becuase of this, OSM created the Trial Stewardship Initative in 2021, as a mission for volunteers to correctly tag the trails in national parks accordingly and encourage the applications to display the trails differntly. At the moment, this tagging is done by hand and tagging an entire park for a team takes months. This report is hoping to find ways to automate this tagging, starting with Ohio using existing data from OSM's overpass API and ODNR's(Ohio Department of National Recources) ArcGIS REST Service. </p>

<p>For more information on the Trail Stewardship Initiative this presentation by OSM is availilbe: <a>https://openstreetmap.us/events/state-of-the-map-us/2024/osm-us-trails-stewardship-initiative/</a></p>

<h2>Gathering data</h2>

<H4>OSM data</H4>
I entered this search in the turbo wizard on https://overpass-turbo.eu/, Overpass being Open Street Maps API in order to revieve all listed paths in Ohio

/*
This has been generated by the overpass-turbo wizard.
The original search was:
“highway=path in "Ohio"”
*/
[out:json][timeout:25];
// fetch area “Ohio” to search in
{{geocodeArea:Ohio}}->.searchArea;
// gather results
nwr["highway"="path"](area.searchArea);
// print results
out geom;

I then rendered this file using jupyterlab-geojson 3.4.0

<h4>ONDR data</h4>
Entered an SQL Query to https://gis2.ohiodnr.gov/arcgis/rest/services/OIT_Services/Trails_WayPoints_POI/MapServer/2

Where: 1=1

Out Fields: *

Return Geometry: true

Format: GEOjson

resultOffset: 0

resultRecordCount: 1000

This data is used in Ohio official Trial hiking app, and should serve as a good source for the offical and maintained trails


In [1]:
geojson_file = 'export.geojson'

ONDRgeojson_file = 'query.geojson'

In [2]:
import urllib
import requests
import pandas as pd
import json
import matplotlib.pyplot as plt
import base64
from IPython.display import GeoJSON
with open('export.geojson') as fo:
    data_str = fo.read()
    data = json.loads(data_str) 
df = pd.DataFrame(data)
import geopandas as gpd

In [3]:
export_gdf = gpd.read_file("export.geojson")
query_gdf = gpd.read_file("query.geojson")

<h2>Cleaning Data</h2>
Part of the issue the Trail Stewardship Initiative is trying to tackle is that OSM does not have a very percise standard for the differntiating between the type of ways they collect, just that they exist. So, with the given keys, i'll have to try to select the several that could refer to the trail being an offical and maintained Ohio trail, and remove the rest ODNR's dataset has quality information and can serve as what we should expect our map of offical OSM trails to look like.

In [4]:
#dataframe of OSM data
#export_gdf.head(1)

In [5]:
#dataframe of ODNR's data
#query_gdf.head(1)

In [6]:
#ODNR columns 
query_gdf.columns.tolist()

['OBJECTID',
 'TRAIL_NAME',
 'LENGTH',
 'USE_CODE',
 'STATUS',
 'DIFF_LEVEL',
 'ACCESS',
 'SURF_TYPE',
 'DIVISION',
 'LANDS_NAME',
 'RuleID_2',
 'Shape.STLength()',
 'Trails_jnFld',
 'Total_Length',
 'geometry']

In [7]:
#OSM columns, many only have a few features using them
export_gdf.columns.tolist()

['id',
 '@id',
 'FIXME',
 'PDF_lineColor',
 'Trail',
 'abandoned',
 'abandoned:highway',
 'abandoned:railway',
 'access',
 'access:conditional',
 'access:disabled',
 'alt_name',
 'amenity',
 'area',
 'athletics',
 'atv',
 'bicycle',
 'bicycle_road',
 'bridge',
 'bridge:name',
 'bridge:ref',
 'bridge:structure',
 'capacity',
 'check_date',
 'check_date:surface',
 'comment',
 'construction',
 'covered',
 'created_by',
 'crossing',
 'crossing:island',
 'crossing:markings',
 'cutting',
 'cycleway',
 'description',
 'disused:highway',
 'disused:leisure',
 'dog',
 'ele',
 'embankment',
 'fee',
 'fixme',
 'foot',
 'footway',
 'ford',
 'golf',
 'golf_cart',
 'handrail',
 'heritage',
 'highway',
 'hiking',
 'horse',
 'horse_scale',
 'incline',
 'informal',
 'key',
 'landuse',
 'lanes',
 'layer',
 'lcn',
 'lcn_ref',
 'leisure',
 'length',
 'length_unit',
 'level',
 'line',
 'lit',
 'man_made',
 'manufacturer',
 'material',
 'maxspeed',
 'maxweight',
 'maxwidth',
 'motor_vehicle',
 'motorcar',
 '

In [8]:
# Count unique operators and their frequencies
operator_counts = export_gdf['operator'].value_counts()

# Display the counts
print(operator_counts)

operator
Cincinnati Off-Road Alliance            33
Ohio Department of Natural Resources     7
Wayne National Forest                    6
ODNR                                     3
Western Wildlife Corridor                1
Cleveland Metroparks                     1
Name: count, dtype: int64


In [9]:
filtered_export_gdf = export_gdf[export_gdf['operator'].notna() & (export_gdf['operator'] != "")]
#filtered_export_gdf.head()

In [10]:
# Count unique operators and their frequencies
official_name_counts = export_gdf['official_name'].value_counts()

# Display the counts
print(official_name_counts)

official_name
North Country Trail-Athens East       12
North Country Trail-Athens Central     1
Name: count, dtype: int64


In [11]:
# Filter DataFrame for specific operator values
filtered_gdf_Offname = export_gdf[export_gdf['official_name'].isin(['North Country Trail-Athens East', 'North Country Trail-Athens Central'])]

# Display the filtered DataFrame
#filtered_gdf_Offname.head()

In [12]:
# Count unique operators and their frequencies
formal_counts = export_gdf['informal'].value_counts()

# Display the counts
print(formal_counts)

informal
yes    304
no       7
Name: count, dtype: int64


In [13]:
# Filter DataFrame for specific operator values
filtered_gdf_informal = export_gdf[export_gdf['informal'].isin(['no'])]

# Display the filtered DataFrame
#filtered_gdf_informal.head()

In [14]:
# Count unique operators and their frequencies
access_counts = export_gdf['access'].value_counts()

# Display the counts
print(access_counts)

access
private        424
no             117
permissive     107
yes             56
customers       40
unknown          4
destination      2
designated       1
discouraged      1
Name: count, dtype: int64


In [15]:
# Filter DataFrame for specific operator values
filtered_gdf_access = export_gdf[export_gdf['access'].isin(['permissive', 'yes'])]

# Display the filtered DataFrame
#filtered_gdf_access.head()

In [16]:
# Count unique operators and their frequencies
ref_counts = export_gdf['ref'].value_counts()

# Display the counts
print(ref_counts)

ref
BT          30
TR 1024     12
GO           9
TR 1001      2
CH 47        2
T-286        1
TR 173       1
TR 1029      1
SR 56        1
TR 1002      1
TR 1001A     1
Name: count, dtype: int64


In [17]:
filtered_export_gdf2 = export_gdf[export_gdf['ref'].notna() & (export_gdf['ref'] != "")]

In [18]:
# Concatenate the DataFrames row-wise (stacking them)
merged_gdf = pd.concat([filtered_gdf_access, filtered_gdf_Offname, filtered_export_gdf, filtered_export_gdf2,filtered_gdf_informal], axis=0, ignore_index=True)

# Display the merged DataFrame
#merged_gdf.head(100)

<h4>Comparing Cleaned Data to Offical Data</h4>
After combing through the OSM data and combining all of the keys that could refer to the data being Offcial, I rendered it using jupyterlab-geojson 3.4.0 as a good way to visual compare the data. 

<h4>Cleaned OSM data</h4>

In [19]:
#GeoJSON(data=merged_gdf.to_json())

/opt/anaconda3/envs/data_fa24/lib/python3.12/site-packages/IPython/core/display.py:618: UserWarning: JSON expects JSONable dict or list, not JSON strings
  warnings.warn("JSON expects JSONable dict or list, not JSON strings")


<IPython.display.GeoJSON object>

<h4>Official ODNR data</h4>

In [20]:
ONDRgeojson_file = 'query.geojson'
# Load the GeoJSON file
with open(ONDRgeojson_file, 'r') as f:
    geojson_data = json.load(f)

# Render the GeoJSON
#GeoJSON(data=geojson_data)

<IPython.display.GeoJSON object>

While the cleaned OSM data is a little light, it matches the pattern of the offical ODNR data pretty well.

<h2>Sorting trails and creating a map</h2>

OSM's ways, the lines between two geographicla points, are tagged with information. The tag the indeicates something is a trail, is Highway = path, export.geojson contains every way listed as Highway = Path in Ohio. If the trail has other tags, it can help indicate what kind of trail. If it has a name and is not listed as informal, it is likely and offical path, if it is informal, then it is likely as social trail or in otherwords, and trail used by people but not intended by the park. If Access is tagged as no then the trail is closed or private, if the path only has the tag Highway = Path, then it is likely not part of a maintained park system. I then organized and color coded this data set of trails based on the Trails Stewardship Initiative recomended nomenclature using folium, as the previous render allowed the user to click on the way, but not customize color.

<img src="https://wiki.openstreetmap.org/w/images/thumb/a/af/Potential_Trail_Rendering_Decision_Tree.png/700px-Potential_Trail_Rendering_Decision_Tree.png" alt="Potential Trail Rendering Tree" />

In [25]:
import folium
from folium.plugins import FloatImage

# Path to your GeoJSON file
geojson_file = 'export.geojson'

# Load the GeoJSON data
with open(geojson_file, 'r') as file:
    geojson_data = json.load(file)

# Define a function to style the GeoJSON features dynamically
def style_function(feature):
    properties = feature.get('properties', {})
    # Designated Trails
    # Check for an operator that is not null
    if properties.get('operator') is not None:
        return {'color': '#1f78b4'}  # Blue
    
    # Check for an official name that is not null
    elif properties.get('official name') is not None:
        return {'color': '#1f78b4'}  # Blue

    # Check for access that equals 'permissive' or 'yes'
    elif properties.get('access') in ['permissive', 'yes']:
        return {'color': '#1f78b4'}  # Blue

    # Check for a ref that is not null
    elif properties.get('ref') is not None:
        return {'color': '#1f78b4'}  # Blue
        
    elif properties.get('informal') in ['no']:
        return {'color': '#1f78b4'}  # Blue

    # Informal Trails
    elif properties.get('informal') in ['yes']:
        return {'color': '#00ff00'}  # Green
        
    # Non-Public Trails
    elif properties.get('access') in ['private', 'no']:
        return {'color': '#ff000d'}  # Red

    # Unclear
    # Default style for other trails
    return {'color': '#D3D3D3'}  # Blue
# Initialize a map centered on Ohio
m = folium.Map(location=[40.3675, -82.9962], zoom_start=8, tiles="cartodb positron")

# Add the GeoJSON data to the map with the style function
folium.GeoJson(
    geojson_data,
    style_function=style_function
).add_to(m)

# Add a legend (HTML and CSS)
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 200px;
    height: 150px;
    background-color: white;
    border:2px solid grey;
    z-index:9999;
    font-size:14px;
    padding: 10px;
">
    <b>Trail Legend</b><br>
    <i style="background: #1f78b4; width: 18px; height: 18px; display: inline-block;"></i> Designated Trail<br>
    <i style="background: #00ff00; width: 18px; height: 18px; display: inline-block;"></i> Informal Trail<br>
    <i style="background: #ff000d; width: 18px; height: 18px; display: inline-block;"></i> Non-Public Trail<br>
    <i style="background: #D3D3D3; width: 18px; height: 18px; display: inline-block;"></i> Other Trail
</div>
"""
#legend 

m.get_root().html.add_child(folium.Element(legend_html))

# Save to an HTML file or display in a Jupyter Notebook
#m

<h2>Conclusion</h2>
With this data, we can gain a better understanding of the current state of tagging for Ohio trails on Open Street Maps, and where it might need to improve. Part of the issue is that OSM is a forever ongoing project that might lack the specific and marked data that a complete source like ODNR could provide. Manual re-tagging might need to be done to correct missed tagged and umarked trails. We also have a solid structure for trail apps to use when deciphering what data from OSM to recomend to users as the trails they can trust to Hike.